# **Universal Notebook Environment Setup**

In [1]:
import os
import sys
import wandb

# --- AUTOMATIC ENVIRONMENT SETUP ---
KAGGLE_RUN = os.path.exists('/kaggle/working')

if KAGGLE_RUN:
    print("Running on Kaggle. Setting up paths...")
    # Kaggle Secret for W&B
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

    # Create a symlink so /content/artifacts works on Kaggle
    os.makedirs('/content', exist_ok=True)
    if not os.path.exists('/content/artifacts'):
        # Map Kaggle input artifacts to Colab path
        os.system('ln -s /kaggle/input /content/artifacts')
else:
    print("Running on Google Colab.")
    # Colab Secret for W&B
    try:
        from google.colab import userdata
        wandb_api_key = userdata.get('WANDB_API_KEY')
        wandb.login(key=wandb_api_key)
    except Exception as e:
        print("W&B Secret not found, you may need to login manually.")
        wandb.login()

Running on Kaggle. Setting up paths...


# **Fetch Augmented Images and Model From W&B**

In [2]:
import os
import sys
import wandb

# Initialize a single W&B run for environment setup
run = wandb.init(project="pcb-defect-detection", job_type="setup")

print("--- Downloading Dataset Artifact ---")
# 1. Download Dataset
artifact_dataset = run.use_artifact('pcb-augmented-dataset:v1', type='dataset')
dataset_dir = artifact_dataset.download()
print(f"Dataset ready at: {os.path.abspath(dataset_dir)}")

print("\n--- Downloading Model Code Artifact ---")
# 2. Download Model Code (pcb_model.py)
artifact_code = run.use_artifact('pcb-model-code:latest', type='code')
code_dir = artifact_code.download()

# Add to sys.path to allow immediate import
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)
print(f"Model code ready at: {code_dir}")

# Optional: verify import
from pcb_model import HybridDetector
print("HybridDetector class successfully imported.")

run.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: naufalsatya (nsp-deep-learning-projects) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


--- Downloading Dataset Artifact ---


wandb: Downloading large artifact 'pcb-augmented-dataset:v1', 269.55MB. 5544 files...
wandb:   5544 of 5544 files downloaded.  
Done. 00:00:43.1 (6.3MB/s)


Dataset ready at: /kaggle/working/artifacts/pcb-augmented-dataset:v1

--- Downloading Model Code Artifact ---


wandb:   1 of 1 files downloaded.  


Model code ready at: /kaggle/working/artifacts/pcb-model-code:v0


Cloning into 'LeYOLO'...


HybridDetector class successfully imported.


# **Device Setup & W&B Tracking Initialization**

In [3]:
!pip install -q wandb timm opencv-python albumentations

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import wandb
import timm
from torch.utils.data import Dataset, DataLoader

# Initialize the experiment tracking run
run = wandb.init(
    project="pcb-defect-detection",
    name="Proper_Model_Training_FIXED",
    notes="Training the hybrid architecture and metric logging (chore: add new functions and fix parameters)",
    config={
        "batch_size": 32,
        "epochs": 150,
        "image_size": 480
    }
)
config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# **Data Loader From W&B**

In [9]:
!pip install -q ultralytics

In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

class PCBDataset(Dataset):
    def __init__(self, img_dir, label_dir, img_size=480):
        self.img_dir   = img_dir
        self.label_dir = label_dir
        self.img_size  = img_size
        self.img_names = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        # Load and preprocess image
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))

        # Normalize and convert to tensor (CHW format)
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0

        # Load YOLO format labels
        label_path = os.path.join(self.label_dir, self.img_names[idx].replace('.jpg', '.txt'))
        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        labels.append(int(float(parts[0])))  # handles '5.0' format
                        boxes.append([float(x) for x in parts[1:]])

        targets = {
            "boxes":   torch.tensor(boxes,  dtype=torch.float32),
            "labels":  torch.tensor(labels, dtype=torch.int64),
            "raw_img": img  # kept for W&B visualization
        }
        return img_tensor, targets


def collate_fn(batch, device):
    images   = torch.stack([item[0] for item in batch])
    raw_imgs = [item[1]["raw_img"] for item in batch]

    batch_idx_list, cls_list, box_list = [], [], []

    for b_idx, item in enumerate(batch):
        targets   = item[1]
        num_boxes = len(targets["labels"])
        if num_boxes > 0:
            batch_idx_list.append(torch.full((num_boxes,), b_idx, dtype=torch.long))
            cls_list.append(targets["labels"].unsqueeze(1))
            box_list.append(targets["boxes"])

    # Fixed: plain torch.cat — no broken markdown hyperlinks
    if len(batch_idx_list) > 0:
        batch_dict = {
            'batch_idx': torch.cat(batch_idx_list, dim=0).to(device),
            'cls':       torch.cat(cls_list,       dim=0).to(device),
            'bboxes':    torch.cat(box_list,        dim=0).to(device)
        }
    else:
        batch_dict = {
            'batch_idx': torch.empty(0,       dtype=torch.long).to(device),
            'cls':       torch.empty((0, 1),  dtype=torch.long).to(device),
            'bboxes':    torch.empty((0, 4),  dtype=torch.float32).to(device)
        }

    return images, batch_dict, raw_imgs


# DATASET & DATALOADER SETUP 
DATA_DIR     = "/kaggle/working/artifacts/pcb-augmented-dataset:v1/train"
full_dataset = PCBDataset(
    os.path.join(DATA_DIR, "images"),
    os.path.join(DATA_DIR, "labels")
)

# 80/20 train/val split — fixed seed for reproducibility
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size   = total_size - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# collate_fn now receives device via lambda — no global dependency
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, device)
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,  # order doesn't matter for evaluation
    collate_fn=lambda b: collate_fn(b, device)
)

print(f"Dataset split — Train: {train_size} | Val: {val_size}")

Dataset split — Train: 2217 | Val: 555


# **Import Hybrid Model Architecture**

In [13]:
from pcb_model import HybridDetector

# Initialize the model using the external file
model = HybridDetector(num_classes=6, image_size=config.image_size).to(device)
print("Model successfully imported from pcb_model.py and initialized.")

model.safetensors:   0%|          | 0.00/5.54M [00:00<?, ?B/s]

Model successfully imported from pcb_model.py and initialized.


# **Custom Model Evaluation Function**

In [ ]:
import torch
import torchvision.ops as ops
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def evaluate_model(
    model,
    val_loader,
    device,
    conf_threshold: float = 0.05,  # Fixed from 0.10 — threshold sweep proved 0.05 is optimal
    iou_threshold: float = 0.45,
) -> dict:
    model.eval()

    # Local instance — no global state pollution
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox', class_metrics=True)  # added class_metrics=True

    with torch.no_grad():
        for images, target_dict, _ in val_loader:
            images = images.to(device)

            # Dynamic sizing — works for any resolution
            _, _, H, W = images.shape

            decoded_preds = model(images)  # [B, 4+NC, Anchors]

            preds_list, targets_list = [], []

            for b in range(images.shape[0]):
                # GROUND TRUTH 
                gt_mask = target_dict['batch_idx'] == b
                gt_boxes_norm = target_dict['bboxes'][gt_mask]  # normalized cxcywh
                # Explicit dtype — avoids silent mismatch in torchmetrics
                gt_labels = target_dict['cls'][gt_mask].squeeze(-1).to(torch.int64)

                if len(gt_boxes_norm) > 0:
                    x_c, y_c, bw, bh = gt_boxes_norm.unbind(1)
                    gt_boxes_xyxy = torch.stack([
                        (x_c - bw / 2) * W, (y_c - bh / 2) * H,
                        (x_c + bw / 2) * W, (y_c + bh / 2) * H,
                    ], dim=1).to(device)
                else:
                    gt_boxes_xyxy = torch.empty((0, 4), device=device)

                targets_list.append({
                    "boxes":  gt_boxes_xyxy,
                    "labels": gt_labels.to(device),
                })

                # PREDICTIONS + NMS 
                preds = decoded_preds[b]                 # [4+NC, Anchors]
                pred_boxes  = preds[:4, :].T             # [Anchors, 4] cxcywh
                pred_scores = preds[4:, :].T             # [Anchors, NC]

                max_scores, class_indices = pred_scores.max(dim=1)

                conf_mask = max_scores > conf_threshold
                f_boxes  = pred_boxes[conf_mask]
                f_scores = max_scores[conf_mask]
                f_labels = class_indices[conf_mask]

                if len(f_boxes) > 0:
                    x_c, y_c, bw, bh = f_boxes.unbind(1)
                    f_boxes_xyxy = torch.stack([
                        (x_c - bw / 2), (y_c - bh / 2),
                        (x_c + bw / 2), (y_c + bh / 2),
                    ], dim=1)

                    keep = ops.nms(f_boxes_xyxy, f_scores, iou_threshold=iou_threshold)

                    preds_list.append({
                        "boxes":  f_boxes_xyxy[keep],
                        "scores": f_scores[keep],
                        "labels": f_labels[keep].to(torch.int64),
                    })
                else:
                    preds_list.append({
                        "boxes":  torch.empty((0, 4),                   device=device),
                        "scores": torch.empty((0,),                     device=device),
                        "labels": torch.empty((0,), dtype=torch.int64,  device=device),
                    })

            metric.update(preds_list, targets_list)

    results = metric.compute()

    # Restore training mode before returning
    model.train()
    return results

# **Create Visualize Predictions for Validation**

In [ ]:
def visualize_predictions(model, val_loader, device, epoch,
                           conf_threshold=0.05,   # Fixed from 0.20 — consistent with evaluate_model
                           iou_threshold=0.45):
    model.eval()
    with torch.no_grad():
        viz_images, viz_targets, viz_raw = next(iter(val_loader))
        viz_images = viz_images.to(device)
        decoded_preds = model(viz_images)

        viz_img = viz_raw[0].copy()
        h, w, _ = viz_img.shape

        # GROUND TRUTH (Green) 
        gt_mask = viz_targets['batch_idx'] == 0
        for box in viz_targets['bboxes'][gt_mask]:
            x_c, y_c, bw, bh = box.cpu().numpy()
            x1, y1 = int((x_c - bw/2)*w), int((y_c - bh/2)*h)
            x2, y2 = int((x_c + bw/2)*w), int((y_c + bh/2)*h)
            cv2.rectangle(viz_img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # PREDICTIONS (Red) with NMS 
        preds = decoded_preds[0].cpu()
        pred_boxes  = preds[:4, :].T   # [Anchors, 4] cxcywh
        pred_scores = preds[4:, :].T   # [Anchors, NC]

        max_scores, class_indices = pred_scores.max(dim=1)
        conf_mask = max_scores > conf_threshold

        f_boxes   = pred_boxes[conf_mask]
        f_scores  = max_scores[conf_mask]
        f_labels  = class_indices[conf_mask]

        if len(f_boxes) > 0:
            x_c, y_c, bw, bh = f_boxes.unbind(1)
            f_boxes_xyxy = torch.stack([
                (x_c - bw/2), (y_c - bh/2),
                (x_c + bw/2), (y_c + bh/2)
            ], dim=1)
            keep = ops.nms(f_boxes_xyxy, f_scores, iou_threshold=iou_threshold)

            for idx in keep:
                bx1, by1, bx2, by2 = f_boxes_xyxy[idx].numpy()
                score  = f_scores[idx].item()
                cls_id = f_labels[idx].item()  # Added — shows which defect class was detected
                cv2.rectangle(viz_img, (int(bx1), int(by1)), (int(bx2), int(by2)), (255, 0, 0), 2)
                cv2.putText(
                    viz_img,
                    f"cls{cls_id}: {score:.2f}",          # "cls0: 0.87" instead of just "0.87"
                    (int(bx1), int(by1) - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1
                )

    model.train()
    return viz_img

# **Overfit Loop & W&B Training Tracking**

In [ ]:
import torch
import torchvision.ops as ops
from torch.optim.lr_scheduler import CosineAnnealingLR
from ultralytics.utils.loss import v8DetectionLoss

# 1. DUMMY WRAPPER (Required for Hybrid Model + Native Loss) 
class DummyModelConfig:
    def __init__(self, full_model, target_device):
        self._full_model = full_model
        self.device = target_device
        class Args:
            box, cls, dfl, cls_pw = 7.5, 0.5, 1.5, 1.0
        self.args = Args()
        class MockDetectHead:
            def __init__(self, head):
                self.stride, self.nc, self.no, self.device = head.stride, head.nc, head.no, target_device
                self.reg_max = head.ch
                self.use_dfl = True
        self.model = [MockDetectHead(full_model.head)]
    def parameters(self):
        return self._full_model.parameters()

# 2. LOSS, OPTIMIZER & SCHEDULER 
dummy_config = DummyModelConfig(model, device)
yolo_loss_fn = v8DetectionLoss(dummy_config)

epochs = config.epochs
optimizer  = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

# 3. BEST MODEL TRACKING 
# Saves the best weights during training, not just the final epoch
best_map50   = 0.0
best_epoch   = 0

print("Launching Clean Training Run (150 epochs)...")

# 4. TRAINING LOOP 
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_idx, (images, target_dict, raw_imgs) in enumerate(train_loader):
        images = images.to(device)
        optimizer.zero_grad()

        predictions = model(images)
        loss, loss_items = yolo_loss_fn(predictions, target_dict)

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_epoch_loss = epoch_loss / len(train_loader)
    current_lr     = scheduler.get_last_lr()[0]

    wandb.log({
        "Train_Loss":    avg_epoch_loss,
        "Learning_Rate": current_lr,
        "Epoch":         epoch
    })

    # VALIDATION (every 10 epochs and on the final epoch)
    if epoch % 10 == 0 or epoch == epochs - 1:
        print(f"\nRunning Evaluation — Epoch [{epoch}/{epochs}]...")

        # conf_threshold=0.05 — confirmed optimal from threshold sweep
        val_metrics = evaluate_model(model, val_loader, device, conf_threshold=0.05)
        viz_img     = visualize_predictions(model, val_loader, device, epoch, conf_threshold=0.05)

        current_map50 = val_metrics['map_50'].item()

        wandb.log({
            "Val_mAP_50":             current_map50,
            "Val_Recall":             val_metrics['mar_100'].item(),
            "Val_mAP_50_95":          val_metrics['map'].item(),
            "Validation/Predictions": wandb.Image(viz_img, caption=f"Epoch {epoch}"),
            "Epoch":                  epoch
        })

        print(f"Epoch [{epoch}/{epochs}] | Loss: {avg_epoch_loss:.4f} "
              f"| mAP@0.5: {current_map50:.4f} "
              f"| Recall: {val_metrics['mar_100'].item():.4f} "
              f"| LR: {current_lr:.6f}")

        # Best model checkpoint — saves whenever mAP improves, not just at the end
        if current_map50 > best_map50:
            best_map50 = current_map50
            best_epoch = epoch
            torch.save(model.state_dict(), "mobilevit_leyolo_best.pt")
            print(f"  ★ New best model saved — mAP@0.5: {best_map50:.4f} at epoch {best_epoch}")

# 5. SAVE FINAL WEIGHTS
torch.save(model.state_dict(), "mobilevit_leyolo_final.pt")
wandb.save("mobilevit_leyolo_best.pt")
wandb.save("mobilevit_leyolo_final.pt")
wandb.finish()

print(f"\nTraining Complete!")
print(f"  Final weights : mobilevit_leyolo_final.pt")
print(f"  Best weights  : mobilevit_leyolo_best.pt (epoch {best_epoch}, mAP@0.5: {best_map50:.4f})")
print(f"  Use the best weights for deployment — not the final.")

Launching Clean Training Run (150 epochs)...

Running Evaluation — Epoch [0/150]...
Epoch [0/150] | Loss: 869.3049 | mAP@0.5: 0.0000 | Recall: 0.0000 | LR: 0.001000

Running Evaluation — Epoch [10/150]...
Epoch [10/150] | Loss: 134.7766 | mAP@0.5: 0.5274 | Recall: 0.2585 | LR: 0.000987
  ★ New best model saved — mAP@0.5: 0.5274 at epoch 10

Running Evaluation — Epoch [20/150]...
Epoch [20/150] | Loss: 99.5611 | mAP@0.5: 0.5474 | Recall: 0.2668 | LR: 0.000952
  ★ New best model saved — mAP@0.5: 0.5474 at epoch 20

Running Evaluation — Epoch [30/150]...
Epoch [30/150] | Loss: 85.1475 | mAP@0.5: 0.5972 | Recall: 0.2983 | LR: 0.000898
  ★ New best model saved — mAP@0.5: 0.5972 at epoch 30

Running Evaluation — Epoch [40/150]...
Epoch [40/150] | Loss: 76.3434 | mAP@0.5: 0.6099 | Recall: 0.3080 | LR: 0.000827
  ★ New best model saved — mAP@0.5: 0.6099 at epoch 40

Running Evaluation — Epoch [50/150]...
Epoch [50/150] | Loss: 69.5372 | mAP@0.5: 0.6024 | Recall: 0.3128 | LR: 0.000741

Running 

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch [149/150] | Loss: 47.1697 | mAP@0.5: 0.6022 | Recall: 0.3149 | LR: 0.000001


Epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
Learning_Rate,████████▇▇▇▇▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
Train_Loss,█▆▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Val_Recall,▁▇▇█████████████
Val_mAP_50,▁▇▇█████████████
Val_mAP_50_95,▁▆▇█████████████
Epoch,149
Learning_Rate,0.0
Train_Loss,47.16967
Val_Recall,0.31492
Val_mAP_50,0.60223



Training Complete!
  Final weights : mobilevit_leyolo_final.pt
  Best weights  : mobilevit_leyolo_best.pt (epoch 40, mAP@0.5: 0.6099)
  Use the best weights for deployment — not the final.
